In [2]:
import torch
from torch import nn
# https://blog.csdn.net/devil_son1234/article/details/130699031

f:\my_softers\project_IDE\Anconda\envs\jp_layout_pytorch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# torch.save(model.state_dict()),model.state_dict()是一个字典，里面存着我们模型各个部分的参数
# 有时候我们可能希望模型中的某些参数不更新，但又希望参数保存下来
# 此时我们就会用到 register_buffer()

In [3]:
# 方式一：成员变量不会保存在model.state_dict()中
class my_model(nn.Module):
    def __init__(self):
        super(my_model,self).__init__()
        self.conv = nn.Conv2d(1,1,3,1,1)
        self.tensor = torch.randn(size=(1,1,5,5)) # 成员变量
    def forward(self, x):
        return self.conv(x) + self.tensor

x = torch.randn(size=(1,1,5,5))
model = my_model()
model(x)
print(model.state_dict()) # 可见此时 self.tensor不会保存在model.state_dict()
# 既无法保存
print('-'*6)
print(model.tensor.shape)

OrderedDict([('conv.weight', tensor([[[[-0.1909,  0.2738,  0.3246],
          [ 0.2918,  0.0378,  0.2762],
          [ 0.0335, -0.2316,  0.0707]]]])), ('conv.bias', tensor([-0.2065]))])
------
torch.Size([1, 1, 5, 5])


In [7]:
[i for i in model.parameters()]

[Parameter containing:
 tensor([[[[-0.1909,  0.2738,  0.3246],
           [ 0.2918,  0.0378,  0.2762],
           [ 0.0335, -0.2316,  0.0707]]]], requires_grad=True),
 Parameter containing:
 tensor([-0.2065], requires_grad=True)]

In [8]:
model.state_dict()

OrderedDict([('conv.weight',
              tensor([[[[-0.1909,  0.2738,  0.3246],
                        [ 0.2918,  0.0378,  0.2762],
                        [ 0.0335, -0.2316,  0.0707]]]])),
             ('conv.bias', tensor([-0.2065]))])

In [4]:
# 方式二：成员变量并不会随着model.cuda()复制到gpu上
class my_model(nn.Module):
    def __init__(self):
        super(my_model, self).__init__()
        self.conv = nn.Conv2d(1,1,3,1,1)
        self.tensor = torch.randn(size=(1,1,5,5)) # 成员变量
    def forward(self, x):
        return self.conv(x) + self.tensor

x = torch.randn(size=(1,1,5,5))
x = x.to('cuda')
model = my_model().cuda()
model(x)
print(model.state_dict())

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
# 方式三：使用register_buffer(),让成员变量
# 保存在model.state_dict()中，也就是该变量
# 也就是可以随着模型一起通过.cuda()复制到gpu上。
class my_model(nn.Module):
    def __init__(self):
        super(my_model, self).__init__()
        self.conv = nn.Conv2d(1, 1, 3, 1, 1)
        self.tensor = torch.randn(size=(1, 1, 5, 5))
        self.register_buffer('my_buffer', self.tensor)
 
    def forward(self, x):
        return self.conv(x) + self.my_buffer  # 这里不再是self.tensor
 
 
x = torch.randn(size=(1, 1, 5, 5))
x = x.to('cuda')
model = my_model().cuda()
model(x)
print(model.state_dict())
print('..........')
print(model.tensor)


## 总结  
1. 成员变量：不更新，但是不包含在model.state_dict() ! 通过register_buffer()登记过
的张量，会自动成为模型中的参数，随模型移动（gpu/cpu）而移动，但是不会随着梯度进行更新！